# pandas Basics
## MAD 2502 — Applied Data

**Big idea:** A DataFrame lets you inspect, filter, summarize, transform, and communicate tabular data.

By the end, you should be able to load a CSV, inspect a DataFrame, select and filter rows, handle missing values, make grouped summaries, create a graph, and save a result.

> **Lesson rhythm:** Understand → Predict → Run → Change → Use → Practice → Connect


## 1. Understand

The included file `data/campus_events.csv` contains fictional campus events. **One row represents one event.**

Before analysis, always ask:

- What does one row represent?
- What does each column mean?
- Which columns are numeric?
- Are values missing?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

data_path = Path("applied_data/data/campus_events.csv")
if not data_path.exists():
    data_path = Path("data/campus_events.csv")

events = pd.read_csv(data_path)


## 2. Predict

Without running the next cell, predict what each expression tells you:

```python
events.head(3)
events.shape
events.shape[0]
events["category"]
events[["event_name", "attendance"]]
```

Which expression returns a Series? Which returns a DataFrame?


In [ ]:
print("shape:", events.shape)
print("columns:", list(events.columns))
print("category type:", type(events["category"]))
display(events.head(3))
display(events[["event_name", "attendance"]].head(3))


## 3. Run

Now use Boolean conditions to filter rows. Predict the rows selected by each condition before looking at the output.


In [ ]:
large_events = events[events["attendance"] >= 100]
display(large_events[["event_name", "attendance"]])

free_outdoor = events[
    (events["cost"] == 0)
    & (events["location_type"] == "Outdoor")
]
display(free_outdoor[["event_name", "cost", "location_type"]])


Compare your prediction with the result. In pandas, `&` means AND and `|` means OR for element-by-element Boolean conditions. Put each comparison in parentheses.


## 4. Change

Make one controlled change at a time:

1. Change `attendance >= 100` to `attendance >= 150`. Predict whether the result gets longer or shorter.
2. Change AND (`&`) to OR (`|`) in the free/outdoor filter. Predict what changes.
3. Sort attendance ascending instead of descending.


In [ ]:
music_or_sports = events[
    (events["category"] == "Music")
    | (events["category"] == "Sports")
]
display(music_or_sports)

display(
    events.sort_values("attendance", ascending=False)[
        ["event_name", "attendance"]
    ].head(5)
)


## 5. Use

Now answer a real question: **Which event categories tend to have larger attendance?**

First inspect missing values. Then build a grouped summary and a graph.


In [ ]:
print(events.isna().sum())

rating_median = events["rating"].median()
events["rating_filled"] = events["rating"].fillna(rating_median)

category_summary = (
    events.groupby("category", as_index=False)
    .agg(
        number_of_events=("event_name", "count"),
        mean_attendance=("attendance", "mean"),
        median_cost=("cost", "median"),
    )
    .sort_values("mean_attendance", ascending=False)
)

display(category_summary)


Filling a missing rating with the median is a **decision**, not a universal rule. Zero and unknown are not the same thing.


In [ ]:
plot_data = category_summary.sort_values("mean_attendance")
ax = plot_data.plot(
    x="category",
    y="mean_attendance",
    kind="barh",
    legend=False,
    figsize=(7, 4),
)
ax.set(
    xlabel="Mean attendance",
    ylabel="Event category",
    title="Mean attendance by event category",
)
plt.tight_layout()
plt.show()


In [ ]:
output_path = data_path.parent / "category_summary.csv"
category_summary.to_csv(output_path, index=False)
print("saved:", output_path)


## 6. Practice

1. Which event had the highest attendance?
2. How many events were free?
3. What was the median attendance for outdoor events?
4. Which category had the highest average rating?
5. Create a table containing only events with attendance above the overall median.
6. Make one labeled graph that answers a question about the dataset.


## 7. Connect

The pandas workflow is:

**inspect → ask → select → compute → check → communicate**

This appears throughout the capstone. Before making a sophisticated model, make sure you know what one row represents, what is missing, what your filters removed, and what your summary actually means.

**Next connection:** Statistics Villains adds a new challenge—getting the table from a webpage before doing the pandas analysis.
